In [0]:
# Databricks notebook source

# MAGIC %run ./01-config

# COMMAND ----------

class Bronze:

    def __init__(self, catalog="dev"):

        self.conf = Config()

        self.landing_zone = self.conf.base_dir_data + "/raw"
        self.checkpoint_base = self.conf.base_dir_checkpoint + "/checkpoints"

        self.catalog = catalog

        self.bronze_schema = self.conf.bronze_schema
        self.silver_schema = self.conf.silver_schema


    # --------------------------------------------------
    # USER REGISTRATION
    # --------------------------------------------------

    def consume_user_registration(
        self,
        once=True,
        processing_time="5 seconds"
    ):

        from pyspark.sql import functions as F

        schema = """
            user_id long,
            device_id long,
            mac_address string,
            registration_timestamp double
        """

        df_stream = (
            spark.readStream
            .format("cloudFiles")
            .schema(schema)
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option(
                "maxFilesPerTrigger",
                self.conf.maxFilesPerTrigger
            )
            .load(
                self.landing_zone +
                "/registered_users_bz"
            )
            .withColumn(
                "load_time",
                F.current_timestamp()
            )
            .withColumn(
                "source_file",
                F.input_file_name()
            )
        )

        stream_writer = (
            df_stream.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                self.checkpoint_base +
                "/registered_users_bz"
            )
            .outputMode("append")
            .queryName(
                "registered_users_bz_ingestion_stream"
            )
        )

        target_table = (
            f"{self.catalog}."
            f"{self.bronze_schema}."
            f"registered_users_bz"
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .toTable(target_table)
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .toTable(target_table)
        )


    # --------------------------------------------------
    # GYM LOGINS
    # --------------------------------------------------

    def consume_gym_logins(
        self,
        once=True,
        processing_time="5 seconds"
    ):

        from pyspark.sql import functions as F

        schema = """
            mac_address string,
            gym bigint,
            login double,
            logout double
        """

        df_stream = (
            spark.readStream
            .format("cloudFiles")
            .schema(schema)
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option(
                "maxFilesPerTrigger",
                self.conf.maxFilesPerTrigger
            )
            .load(
                self.landing_zone +
                "/gym_logins_bz"
            )
            .withColumn(
                "load_time",
                F.current_timestamp()
            )
            .withColumn(
                "source_file",
                F.input_file_name()
            )
        )

        stream_writer = (
            df_stream.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                self.checkpoint_base +
                "/gym_logins_bz"
            )
            .outputMode("append")
            .queryName(
                "gym_logins_bz_ingestion_stream"
            )
        )

        target_table = (
            f"{self.catalog}."
            f"{self.bronze_schema}."
            f"gym_logins_bz"
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .toTable(target_table)
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .toTable(target_table)
        )


    # --------------------------------------------------
    # KAFKA MULTIPLEX
    # --------------------------------------------------

    def consume_kafka_multiplex(
        self,
        once=True,
        processing_time="5 seconds"
    ):

        from pyspark.sql import functions as F

        schema = """
            key string,
            value string,
            topic string,
            partition bigint,
            offset bigint,
            timestamp bigint
        """

        df_date_lookup = (
            spark.table(
                f"{self.catalog}."
                f"{self.silver_schema}."
                f"date_lookup"
            )
            .select(
                "date",
                "week_part"
            )
        )

        df_stream = (
            spark.readStream
            .format("cloudFiles")
            .schema(schema)
            .option(
                "cloudFiles.format",
                "json"
            )
            .option(
                "maxFilesPerTrigger",
                self.conf.maxFilesPerTrigger
            )
            .load(
                self.landing_zone +
                "/kafka_multiplex_bz"
            )
            .withColumn(
                "load_time",
                F.current_timestamp()
            )
            .withColumn(
                "source_file",
                F.input_file_name()
            )
            .join(
                F.broadcast(df_date_lookup),
                F.to_date(
                    (
                        F.col("timestamp") / 1000
                    ).cast("timestamp")
                ) == F.col("date"),
                "left"
            )
        )

        stream_writer = (
            df_stream.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                self.checkpoint_base +
                "/kafka_multiplex_bz"
            )
            .outputMode("append")
            .queryName(
                "kafka_multiplex_bz_ingestion_stream"
            )
        )

        target_table = (
            f"{self.catalog}."
            f"{self.bronze_schema}."
            f"kafka_multiplex_bz"
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .toTable(target_table)
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .toTable(target_table)
        )


    # --------------------------------------------------
    # RUN ALL BRONZE INGESTIONS
    # --------------------------------------------------

    def consume(
        self,
        once=True,
        processing_time="5 seconds"
    ):

        import time

        start = int(time.time())

        print(
            "\nStarting bronze layer consumption..."
        )

        self.consume_user_registration(
            once,
            processing_time
        )

        self.consume_gym_logins(
            once,
            processing_time
        )

        self.consume_kafka_multiplex(
            once,
            processing_time
        )

        if once:
            for stream in spark.streams.active:
                stream.awaitTermination()

        print(
            f"Completed bronze layer consumption "
            f"in {int(time.time()) - start} seconds"
        )


    # --------------------------------------------------
    # GENERIC VALIDATION
    # --------------------------------------------------

    def assert_count(
        self,
        table_name,
        expected_count,
        filter_condition="true"
    ):

        print(
            f"Validating record counts "
            f"in {table_name}...",
            end=""
        )

        actual_count = (
            spark.read
            .table(
                f"{self.catalog}."
                f"{self.bronze_schema}."
                f"{table_name}"
            )
            .where(filter_condition)
            .count()
        )

        assert actual_count == expected_count, (
            f"Expected {expected_count:,} records, "
            f"found {actual_count:,} "
            f"in {table_name} "
            f"where {filter_condition}"
        )

        print(
            f"Found {actual_count:,} / "
            f"Expected {expected_count:,}: Success"
        )


    # --------------------------------------------------
    # VALIDATE BRONZE
    # --------------------------------------------------

    def validate(self, sets=1):

        import time

        start = int(time.time())

        print(
            "\nValidating bronze layer records..."
        )

        self.assert_count(
            "registered_users_bz",
            5 if sets == 1 else 10
        )

        self.assert_count(
            "gym_logins_bz",
            8 if sets == 1 else 16
        )

        self.assert_count(
            "kafka_multiplex_bz",
            7 if sets == 1 else 13,
            "topic = 'user_info'"
        )

        self.assert_count(
            "kafka_multiplex_bz",
            16 if sets == 1 else 32,
            "topic = 'workout'"
        )

        self.assert_count(
            "kafka_multiplex_bz",
            sets * 253801,
            "topic = 'bpm'"
        )

        print(
            f"Bronze layer validation completed "
            f"in {int(time.time()) - start} seconds"
        )